**Import du dataset depuis hunginface**

In [1]:
%mkdir medical_bios_data
!wget -O medical_bios_data/bios.zip "https://huggingface.co/datasets/coastalcph/medical-bios/resolve/main/bios.zip"

--2026-04-07 14:15:11--  https://huggingface.co/datasets/coastalcph/medical-bios/resolve/main/bios.zip
Resolving huggingface.co (huggingface.co)... 18.65.25.19, 18.65.25.54, 18.65.25.103, ...
Connecting to huggingface.co (huggingface.co)|18.65.25.19|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6523dbfaab141659412a22b6/405d2dae7c20a24f6a4d0d1deba1cb783a535213ab2652d1cab958b4fc11d94b?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27bios.zip%3B+filename%3D%22bios.zip%22%3B&response-content-type=application%2Fzip&Expires=1775574911&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzc1NTc0OTExfX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjUyM2RiZmFhYjE0MTY1OTQxMmEyMmI2LzQwNWQyZGFlN2MyMGEyNGY2YTRkMGQxZGViYTFjYjc4M2E1MzUyMTNhYjI2NTJkMWNhYjk1OGI0ZmMxMWQ5NGJcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29udGVudC10eXBlPSoifV19&Signature=BD

In [2]:
# Étape 4a: Extraire le fichier zip
import zipfile
import os

zip_path = 'medical_bios_data/bios.zip'
extract_path = 'medical_bios_data/'

# Vérifier que le zip existe 
if os.path.exists(zip_path):
    print(f"✅ Fichier zip trouvé: {zip_path}")
    print(f"📁 Taille: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")
    
    # Extraire le contenu
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
        print("✅ Extraction terminée!")
        
        # Lister les fichiers dans le zip
        print("\n📋 Contenu du zip:")
        for file_info in zip_ref.infolist():
            print(f"  - {file_info.filename} ({file_info.file_size} bytes)")
            
else:
    print(f"❌ Fichier zip non trouvé: {zip_path}")

✅ Fichier zip trouvé: medical_bios_data/bios.zip
📁 Taille: 1.82 MB
✅ Extraction terminée!

📋 Contenu du zip:
  - test_rationales.jsonl (300522 bytes)
  - __MACOSX/._test_rationales.jsonl (176 bytes)
  - test.jsonl (946345 bytes)
  - __MACOSX/._test.jsonl (275 bytes)
  - train.jsonl (7595575 bytes)
  - __MACOSX/._train.jsonl (219 bytes)
  - validation.jsonl (938592 bytes)
  - __MACOSX/._validation.jsonl (219 bytes)


In [3]:
# Étape 4b: Vérifier le contenu extrait
import os

# Lister tous les fichiers dans le dossier medical_bios_data
print("📂 Contenu du dossier medical_bios_data/:")
for root, dirs, files in os.walk('medical_bios_data/'):
    for file in files:
        file_path = os.path.join(root, file)
        file_size = os.path.getsize(file_path)
        print(f"  📄 {file_path} ({file_size:,} bytes)")

# Vérifier spécifiquement les fichiers JSONL
jsonl_files = ['train.jsonl', 'validation.jsonl', 'test.jsonl']
print(f"\n🔍 Recherche des fichiers JSONL:")

for jsonl_file in jsonl_files:
    # Chercher dans le dossier et sous-dossiers
    found = False
    for root, dirs, files in os.walk('medical_bios_data/'):
        if jsonl_file in files:
            full_path = os.path.join(root, jsonl_file)
            size = os.path.getsize(full_path)
            print(f"  ✅ {full_path} ({size:,} bytes)")
            found = True
            break
    if not found:
        print(f"  ❌ {jsonl_file} non trouvé")

📂 Contenu du dossier medical_bios_data/:
  📄 medical_bios_data/train.jsonl (7,595,575 bytes)
  📄 medical_bios_data/test_rationales.jsonl (300,522 bytes)
  📄 medical_bios_data/validation.jsonl (938,592 bytes)
  📄 medical_bios_data/bios.zip (1,912,483 bytes)
  📄 medical_bios_data/test.jsonl (946,345 bytes)
  📄 medical_bios_data/__MACOSX/._train.jsonl (219 bytes)
  📄 medical_bios_data/__MACOSX/._validation.jsonl (219 bytes)
  📄 medical_bios_data/__MACOSX/._test.jsonl (275 bytes)
  📄 medical_bios_data/__MACOSX/._test_rationales.jsonl (176 bytes)

🔍 Recherche des fichiers JSONL:
  ✅ medical_bios_data/train.jsonl (7,595,575 bytes)
  ✅ medical_bios_data/validation.jsonl (938,592 bytes)
  ✅ medical_bios_data/test.jsonl (946,345 bytes)


**chargement des datasets**

In [12]:
# Étape 5: Charger les données JSONL et explorer leur structure
import json
import pandas as pd

def load_jsonl(file_path):
    """Charger un fichier JSONL en liste de dictionnaires"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():  # Ignorer les lignes vides
                data.append(json.loads(line))
    return data

# Charger les datasets
print("🔄 Chargement des datasets...")

# Commencer par charger quelques exemples du train pour voir la structure
train_sample = []
with open('medical_bios_data/train.jsonl', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5:  # Charger seulement les 5 premiers pour l'exploration
            break
        if line.strip():
            train_sample.append(json.loads(line))

print(f"✅ Échantillon chargé: {len(train_sample)} exemples")
print(f"\n📋 Structure d'un exemple:")
print(json.dumps(train_sample[0], indent=2, ensure_ascii=False))

# Regarder les clés disponibles
if train_sample:
    keys = train_sample[0].keys()
    print(f"\n🔑 Clés disponibles: {list(keys)}")

🔄 Chargement des datasets...
✅ Échantillon chargé: 5 exemples

📋 Structure d'un exemple:
{
  "full_text": "Dr. Vikram Prasad is an experienced Dentist in Sowkhya Ayurveda Speciality Clinic, Chennai. He has been a practicing Dentist for 20 years. He has done BDS . He is currently associated with Sree Sai Dental Clinic in Sowkhya Ayurveda Speciality Clinic, Chennai. Book an appointment online with Dr. Vikram Prasad and consult privately on Lybrate.com.",
  "text": "He has been a practicing Dentist for 20 years. He has done BDS . He is currently associated with Sree Sai Dental Clinic in Sowkhya Ayurveda Speciality Clinic, Chennai. Book an appointment online with Dr. Vikram Prasad and consult privately on Lybrate.com.",
  "title": "dentist",
  "gender": "Male",
  "date": "2018-09"
}

🔑 Clés disponibles: ['full_text', 'text', 'title', 'gender', 'date']


In [13]:
#chargement du train_set avec la fonction load_jsonl
train_set = load_jsonl('medical_bios_data/train.jsonl')
print(f"✅ Train set chargé: {len(train_set)} exemples") 

✅ Train set chargé: 8000 exemples


In [14]:
# Étape 6a: Analyse de la complétude des données
import pandas as pd

# Convertir en DataFrame pour l'analyse
df_train = pd.DataFrame(train_set)

print("📊 ANALYSE DE LA QUALITÉ DES DONNÉES")
print("=" * 50)

# 1. Vérifier les valeurs manquantes
print("🔍 1. VALEURS MANQUANTES:")
missing_analysis = {}
for column in ['full_text', 'text', 'title', 'gender', 'date']:
    # Compter les valeurs None, vides, ou NaN
    null_count = df_train[column].isnull().sum()
    empty_count = (df_train[column] == '').sum()
    total_missing = null_count + empty_count
    percentage = (total_missing / len(df_train)) * 100
    
    missing_analysis[column] = {
        'null': null_count,
        'empty': empty_count, 
        'total_missing': total_missing,
        'percentage': percentage
    }
    
    status = "✅" if total_missing == 0 else "⚠️"
    print(f"  {status} {column}: {total_missing}/{len(df_train)} manquants ({percentage:.2f}%)")

print(f"\n📈 Taille du dataset: {len(df_train)} exemples")
print(f"📋 Colonnes: {list(df_train.columns)}")

📊 ANALYSE DE LA QUALITÉ DES DONNÉES
🔍 1. VALEURS MANQUANTES:
  ✅ full_text: 0/8000 manquants (0.00%)
  ⚠️ text: 5/8000 manquants (0.06%)
  ✅ title: 0/8000 manquants (0.00%)
  ✅ gender: 0/8000 manquants (0.00%)
  ✅ date: 0/8000 manquants (0.00%)

📈 Taille du dataset: 8000 exemples
📋 Colonnes: ['full_text', 'text', 'title', 'gender', 'date']


In [15]:
# Étape 6b: Analyse de la distribution des valeurs
print("\n🔍 2. DISTRIBUTION DES VALEURS:")
print("-" * 30)

# Distribution des genres
print("👥 GENRE:")
gender_counts = df_train['gender'].value_counts()
print(gender_counts)
print(f"Pourcentages: {df_train['gender'].value_counts(normalize=True) * 100}")

# Distribution des titres/professions
print(f"\n🏥 TITRES/PROFESSIONS (Top 10):")
title_counts = df_train['title'].value_counts()
print(title_counts.head(10))

# Analyse des dates
print(f"\n📅 FORMAT DES DATES:")
date_sample = df_train['date'].head(10).tolist()
print(f"Échantillon: {date_sample}")
print(f"Dates uniques: {df_train['date'].nunique()}")

# Longueur des textes
print(f"\n📝 LONGUEUR DES TEXTES:")
df_train['text_length'] = df_train['text'].str.len()
df_train['full_text_length'] = df_train['full_text'].str.len()

print(f"Texte court - Moyenne: {df_train['text_length'].mean():.1f} chars")
print(f"Texte court - Min/Max: {df_train['text_length'].min()} / {df_train['text_length'].max()}")
print(f"Texte complet - Moyenne: {df_train['full_text_length'].mean():.1f} chars") 
print(f"Texte complet - Min/Max: {df_train['full_text_length'].min()} / {df_train['full_text_length'].max()}")


🔍 2. DISTRIBUTION DES VALEURS:
------------------------------
👥 GENRE:
gender
Female    4290
Male      3710
Name: count, dtype: int64
Pourcentages: gender
Female    53.625
Male      46.375
Name: proportion, dtype: float64

🏥 TITRES/PROFESSIONS (Top 10):
title
psychologist    2200
nurse           1638
dentist         1533
physician       1349
surgeon         1280
Name: count, dtype: int64

📅 FORMAT DES DATES:
Échantillon: ['2018-09', '2017-39', '2018-09', '2017-43', '2018-09', '2018-09', '2018-09', '2017-43', '2018-09', '2018-09']
Dates uniques: 6

📝 LONGUEUR DES TEXTES:
Texte court - Moyenne: 375.2 chars
Texte court - Min/Max: 0 / 977
Texte complet - Moyenne: 481.1 chars
Texte complet - Min/Max: 190 / 1000


### Construction du modèle

In [16]:
#chargement des données de validation et test
validation_set = load_jsonl('medical_bios_data/validation.jsonl')
print(f"✅ Validation set chargé: {len(validation_set)} exemples")
test_set = load_jsonl('medical_bios_data/test.jsonl')
print(f"✅ Test set chargé: {len(test_set)} exemples")


df_val = pd.DataFrame(validation_set) 
df_test = pd.DataFrame(test_set)

print(f"📊 TAILLES DES DATASETS:")
print(f"  Train: {len(df_train)} exemples")
print(f"  Validation: {len(df_val)} exemples") 
print(f"  Test: {len(df_test)} exemples")

✅ Validation set chargé: 1000 exemples
✅ Test set chargé: 1000 exemples
📊 TAILLES DES DATASETS:
  Train: 8000 exemples
  Validation: 1000 exemples
  Test: 1000 exemples


In [17]:
# Étape 11c: Nettoyage complet et re-création des données
import re
import pandas as pd

print("🧹 NETTOYAGE COMPLET DES DONNÉES")
print("=" * 40)

# Fonction de nettoyage améliorée
def clean_text_remove_profession_bias(text):
    """
    Remplacer tous les termes de professions et variantes par <PROFESSION>
    """
    cleaned_text = text
    
    # Dictionnaire exhaustif des termes à remplacer
    profession_patterns = [
        # Psychologist variations
        r'\b(psychologist|psychologists|psychology|psychological|therapist|therapists|therapy|counselor|counselors)\b',
        # Nurse variations  
        r'\b(nurse|nurses|nursing|rn|registered nurse|lpn|cna)\b',
        # Dentist variations
        r'\b(dentist|dentists|dental|dentistry|orthodontist|orthodontists|oral)\b', 
        # Physician variations
        r'\b(physician|physicians|doctor|doctors|dr\.?|medical doctor|md|clinician|clinicians)\b',
        # Surgeon variations
        r'\b(surgeon|surgeons|surgery|surgical|operative|operation)\b'
    ]
    
    # Appliquer tous les patterns
    for pattern in profession_patterns:
        cleaned_text = re.sub(pattern, '<PROFESSION>', cleaned_text, flags=re.IGNORECASE)
    
    return cleaned_text

# Appliquer le nettoyage sur tous les datasets
print("🔄 Application du nettoyage...")

# Train
df_train['full_text_cleaned'] = df_train['full_text'].apply(clean_text_remove_profession_bias)
# Validation  
df_val['full_text_cleaned'] = df_val['full_text'].apply(clean_text_remove_profession_bias)
# Test
df_test['full_text_cleaned'] = df_test['full_text'].apply(clean_text_remove_profession_bias)

print("✅ Nettoyage appliqué!")

# Vérification de l'efficacité du nettoyage
print(f"\n📊 VÉRIFICATION DU NETTOYAGE:")
sample_idx = 0
original = df_train['full_text'].iloc[sample_idx]
cleaned = df_train['full_text_cleaned'].iloc[sample_idx]
profession_count = cleaned.count('<PROFESSION>')

print(f"Original: {original}")
print(f"Nettoyé: {cleaned}")
print(f"Nombre de <PROFESSION> trouvés: {profession_count}")

🧹 NETTOYAGE COMPLET DES DONNÉES
🔄 Application du nettoyage...
✅ Nettoyage appliqué!

📊 VÉRIFICATION DU NETTOYAGE:
Original: Dr. Vikram Prasad is an experienced Dentist in Sowkhya Ayurveda Speciality Clinic, Chennai. He has been a practicing Dentist for 20 years. He has done BDS . He is currently associated with Sree Sai Dental Clinic in Sowkhya Ayurveda Speciality Clinic, Chennai. Book an appointment online with Dr. Vikram Prasad and consult privately on Lybrate.com.
Nettoyé: <PROFESSION>. Vikram Prasad is an experienced <PROFESSION> in Sowkhya Ayurveda Speciality Clinic, Chennai. He has been a practicing <PROFESSION> for 20 years. He has done BDS . He is currently associated with Sree Sai <PROFESSION> Clinic in Sowkhya Ayurveda Speciality Clinic, Chennai. Book an appointment online with <PROFESSION>. Vikram Prasad and consult privately on Lybrate.com.
Nombre de <PROFESSION> trouvés: 5


In [18]:
# Étape 7: Préparation des données pour l'embedding (sans split)
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

print("🔄 PRÉPARATION DES DONNÉES POUR L'EMBEDDING")
print("=" * 50)

# 2. Encoder les labels sur TOUT le dataset (train+val+test combined)
le = LabelEncoder()
all_titles = list(df_train['title']) + list(df_val['title']) + list(df_test['title'])
le.fit(all_titles)

# Appliquer l'encodage sur chaque dataset
df_train['title_encoded'] = le.transform(df_train['title'])
df_val['title_encoded'] = le.transform(df_val['title'])
df_test['title_encoded'] = le.transform(df_test['title'])

# 3. Afficher l'encodage des labels
print("\n🏷️ ENCODAGE DES LABELS:")
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
for profession, code in label_mapping.items():
    train_count = (df_train['title'] == profession).sum()
    val_count = (df_val['title'] == profession).sum()
    test_count = (df_test['title'] == profession).sum()
    print(f"  {code}: {profession}")
    print(f"     Train: {train_count}, Val: {val_count}, Test: {test_count}")

# 4. Préparer les données pour l'embedding
print(f"\n📝 PRÉPARATION POUR L'EMBEDDING:")
print(f"  Texte d'entrée: 'full_text_cleaned'")
print(f"  Labels: {len(le.classes_)} classes")
print(f"  Exemple de texte: '{df_train['full_text_cleaned'].iloc[0][:100]}...'")

# Sauvegarder le LabelEncoder pour plus tard
import pickle
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print(f"✅ LabelEncoder sauvegardé dans 'label_encoder.pkl'")

🔄 PRÉPARATION DES DONNÉES POUR L'EMBEDDING

🏷️ ENCODAGE DES LABELS:
  0: dentist
     Train: 1533, Val: 172, Test: 197
  1: nurse
     Train: 1638, Val: 216, Test: 195
  2: physician
     Train: 1349, Val: 172, Test: 170
  3: psychologist
     Train: 2200, Val: 294, Test: 279
  4: surgeon
     Train: 1280, Val: 146, Test: 159

📝 PRÉPARATION POUR L'EMBEDDING:
  Texte d'entrée: 'full_text_cleaned'
  Labels: 5 classes
  Exemple de texte: '<PROFESSION>. Vikram Prasad is an experienced <PROFESSION> in Sowkhya Ayurveda Speciality Clinic, Ch...'
✅ LabelEncoder sauvegardé dans 'label_encoder.pkl'


In [19]:
# Étape 8: Configuration de l'embedding avec DistilRoBERTa
print("🤖 CONFIGURATION DE L'EMBEDDING DISTILROBERTA")
print("=" * 45)

# Installation si nécessaire (pour Google Colab)
!pip install transformers torch --quiet

# Imports pour DistilRoBERTa
from transformers import RobertaTokenizer, RobertaModel
import torch

# Choisir le modèle DistilRoBERTa
model_name = "distilroberta-base"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"📱 Device disponible: {device}")
print(f"🔤 Modèle choisi: {model_name}")

# Charger le tokenizer et le modèle DistilRoBERTa
print("🔄 Chargement du tokenizer et modèle DistilRoBERTa...")
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaModel.from_pretrained(model_name)
model.to(device)
model.eval()  # Mode évaluation pour l'embedding

print("✅ Modèle DistilRoBERTa chargé!")

# Test rapide du tokenizer sur un exemple
sample_text = df_train['full_text_cleaned'].iloc[0]
tokens = tokenizer(sample_text, padding=True, truncation=True, max_length=512, return_tensors="pt")

print(f"\n🔍 TEST DU TOKENIZER DISTILROBERTA:")
print(f"  Texte original: '{sample_text[:100]}...'")
print(f"  Longueur du texte: {len(sample_text)} caractères")
print(f"  Nombre de tokens: {tokens['input_ids'].shape[1]}")
print(f"  Max length configurée: 512 tokens")
print(f"  Tokens (premiers 10): {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0][:10])}")

# Vérifier les dimensions d'embedding
print(f"\n📐 DIMENSIONS D'EMBEDDING:")
print(f"  Vocabulaire size: {tokenizer.vocab_size}")
print(f"  Hidden size: {model.config.hidden_size}")
print(f"  Nombre de layers: {model.config.num_hidden_layers}")
print(f"  Attention heads: {model.config.num_attention_heads}")

🤖 CONFIGURATION DE L'EMBEDDING DISTILROBERTA
📱 Device disponible: cuda
🔤 Modèle choisi: distilroberta-base
🔄 Chargement du tokenizer et modèle DistilRoBERTa...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle DistilRoBERTa chargé!

🔍 TEST DU TOKENIZER DISTILROBERTA:
  Texte original: '<PROFESSION>. Vikram Prasad is an experienced <PROFESSION> in Sowkhya Ayurveda Speciality Clinic, Ch...'
  Longueur du texte: 383 caractères
  Nombre de tokens: 107
  Max length configurée: 512 tokens
  Tokens (premiers 10): ['<s>', '<', 'PR', 'OF', 'ESSION', '>.', 'ĠVik', 'ram', 'ĠPr', 'as']

📐 DIMENSIONS D'EMBEDDING:
  Vocabulaire size: 50265
  Hidden size: 768
  Nombre de layers: 6
  Attention heads: 12


In [20]:
# Étape 9: Fonction pour extraire les embeddings RoBERTa
def get_roberta_embeddings(texts, batch_size=32):
    """
    Extraire les embeddings RoBERTa pour une liste de textes
    """
    model.eval()
    all_embeddings = []
    
    print(f"🔄 Extraction des embeddings pour {len(texts)} textes...")
    
    # Traiter par batch pour éviter les problèmes de mémoire
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenisation du batch
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        # Extraction des embeddings (sans calcul de gradients)
        with torch.no_grad():
            outputs = model(**inputs)
            # Utiliser le [CLS] token (dernière couche, premier token)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(batch_embeddings)
        
        if (i // batch_size + 1) % 10 == 0:
            print(f"  ✅ {i + len(batch_texts)}/{len(texts)} textes traités")
    
    # Concatener tous les embeddings
    final_embeddings = np.concatenate(all_embeddings, axis=0)
    print(f"✅ Extraction terminée! Shape: {final_embeddings.shape}")
    
    return final_embeddings

In [21]:
# Étape 10: Extraction complète des embeddings pour tous les datasets
import numpy as np

print("🚀 EXTRACTION COMPLÈTE DES EMBEDDINGS")
print("=" * 45)

# Préparer les textes pour chaque dataset
train_texts = df_train['full_text_cleaned'].tolist()
val_texts = df_val['full_text_cleaned'].tolist() 
test_texts = df_test['full_text_cleaned'].tolist()

print(f"📊 DATASETS À TRAITER:")
print(f"  Train: {len(train_texts)} textes")
print(f"  Validation: {len(val_texts)} textes")
print(f"  Test: {len(test_texts)} textes")
print(f"  Total: {len(train_texts) + len(val_texts) + len(test_texts)} textes")

# Extraire les embeddings pour chaque dataset
print(f"\n🔄 EXTRACTION EN COURS...")

train_embeddings = get_roberta_embeddings(train_texts)  
val_embeddings = get_roberta_embeddings(val_texts)     
test_embeddings = get_roberta_embeddings(test_texts)

# Préparer les labels
train_labels = df_train['title_encoded'].values
val_labels = df_val['title_encoded'].values 
test_labels = df_test['title_encoded'].values

print(f"\n✅ EXTRACTION TERMINÉE!")
print(f"📐 SHAPES FINALES:")
print(f"  Train: embeddings {train_embeddings.shape}, labels {train_labels.shape}")
print(f"  Valid: embeddings {val_embeddings.shape}, labels {val_labels.shape}")
print(f"  Test: embeddings {test_embeddings.shape}, labels {test_labels.shape}")

# Sauvegarder les embeddings (optionnel, pour éviter de refaire l'extraction)
print(f"\n💾 SAUVEGARDE DES EMBEDDINGS...")
np.save('train_embeddings.npy', train_embeddings)
np.save('val_embeddings.npy', val_embeddings) 
np.save('test_embeddings.npy', test_embeddings)
np.save('train_labels.npy', train_labels)
np.save('val_labels.npy', val_labels)
np.save('test_labels.npy', test_labels)
print(f"✅ Embeddings sauvegardés!")

🚀 EXTRACTION COMPLÈTE DES EMBEDDINGS
📊 DATASETS À TRAITER:
  Train: 8000 textes
  Validation: 1000 textes
  Test: 1000 textes
  Total: 10000 textes

🔄 EXTRACTION EN COURS...
🔄 Extraction des embeddings pour 8000 textes...
  ✅ 320/8000 textes traités
  ✅ 640/8000 textes traités
  ✅ 960/8000 textes traités
  ✅ 1280/8000 textes traités
  ✅ 1600/8000 textes traités
  ✅ 1920/8000 textes traités
  ✅ 2240/8000 textes traités
  ✅ 2560/8000 textes traités
  ✅ 2880/8000 textes traités
  ✅ 3200/8000 textes traités
  ✅ 3520/8000 textes traités
  ✅ 3840/8000 textes traités
  ✅ 4160/8000 textes traités
  ✅ 4480/8000 textes traités
  ✅ 4800/8000 textes traités
  ✅ 5120/8000 textes traités
  ✅ 5440/8000 textes traités
  ✅ 5760/8000 textes traités
  ✅ 6080/8000 textes traités
  ✅ 6400/8000 textes traités
  ✅ 6720/8000 textes traités
  ✅ 7040/8000 textes traités
  ✅ 7360/8000 textes traités
  ✅ 7680/8000 textes traités
  ✅ 8000/8000 textes traités
✅ Extraction terminée! Shape: (8000, 768)
🔄 Extraction d

In [27]:
# Étape 11: Création et entraînement du classifieur
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

print("🤖 CRÉATION ET ENTRAÎNEMENT DU CLASSIFIEUR")
print("=" * 50)

# Charger les embeddings et labels sauvegardés
print("📂 Chargement des données...")
train_embeddings = np.load('train_embeddings.npy')
val_embeddings = np.load('val_embeddings.npy')
test_embeddings = np.load('test_embeddings.npy')
train_labels = np.load('train_labels.npy') 
val_labels = np.load('val_labels.npy')
test_labels = np.load('test_labels.npy')

print(f"✅ Données chargées:")
print(f"  Train: {train_embeddings.shape} -> {train_labels.shape}")
print(f"  Val: {val_embeddings.shape} -> {val_labels.shape}")
print(f"  Test: {test_embeddings.shape} -> {test_labels.shape}")

# Créer le classifieur Random Forest
print(f"\n🌳 Création du classifieur Random Forest...")
rf_classifier = RandomForestClassifier(
    n_estimators=100,       # Moins d'arbres
    max_depth=15,          # Plus faible profondeur  
    min_samples_split=15,  # Plus de contraintes
    min_samples_leaf=5,
    random_state=42,     # Reproductibilité
    n_jobs=-1              # Utiliser tous les cœurs
)

# Entraînement
print(f"🔄 Entraînement en cours...")
rf_classifier.fit(train_embeddings, train_labels)
print(f"✅ Entraînement terminé!")

# Prédictions
print(f"\n🎯 Évaluation des performances...")
train_pred = rf_classifier.predict(train_embeddings)
val_pred = rf_classifier.predict(val_embeddings)

# Métriques globales
train_acc = accuracy_score(train_labels, train_pred)
val_acc = accuracy_score(val_labels, val_pred)

print(f"📊 RÉSULTATS:")
print(f"  Accuracy Train: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"  Accuracy Validation: {val_acc:.4f} ({val_acc*100:.2f}%)")

🤖 CRÉATION ET ENTRAÎNEMENT DU CLASSIFIEUR
📂 Chargement des données...
✅ Données chargées:
  Train: (8000, 768) -> (8000,)
  Val: (1000, 768) -> (1000,)
  Test: (1000, 768) -> (1000,)

🌳 Création du classifieur Random Forest...
🔄 Entraînement en cours...
✅ Entraînement terminé!

🎯 Évaluation des performances...
📊 RÉSULTATS:
  Accuracy Train: 0.9606 (96.06%)
  Accuracy Validation: 0.8110 (81.10%)


In [23]:
# Test avec Logistic Regression (moins prone à l'overfitting)
from sklearn.linear_model import LogisticRegression

print(f"\n📈 TEST AVEC LOGISTIC REGRESSION")
print("=" * 40)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(train_embeddings, train_labels)

lr_train_pred = lr.predict(train_embeddings)
lr_val_pred = lr.predict(val_embeddings)

lr_train_acc = accuracy_score(train_labels, lr_train_pred)
lr_val_acc = accuracy_score(val_labels, lr_val_pred)

print(f"📊 COMPARAISON FINALE:")
print(f"  Random Forest: Train {train_acc*100:.1f}% | Val {val_acc*100:.1f}% | Écart {(train_acc-val_acc)*100:.1f}%")
print(f"  Logistic Reg:  Train {lr_train_acc*100:.1f}% | Val {lr_val_acc*100:.1f}% | Écart {(lr_train_acc-lr_val_acc)*100:.1f}%")

if (train_acc - val_acc) > (lr_train_acc - lr_val_acc):
    print("✅ Confirmation: Random Forest overfitte plus que LogReg")
else:
    print("🤔 Overfitting similaire → problème plus profond")


📈 TEST AVEC LOGISTIC REGRESSION
📊 COMPARAISON FINALE:
  Random Forest: Train 100.0% | Val 81.7% | Écart 18.3%
  Logistic Reg:  Train 85.0% | Val 83.4% | Écart 1.6%
✅ Confirmation: Random Forest overfitte plus que LogReg
